<div style="border-left:14px solid #FF7F50;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#FF7F50;letter-spacing:.08em">EUROPEANA</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">What is inside Europeana?</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Distant reading of a collection: who contributes, what is in it, how to ask the API.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 1 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

# Europeana

The IIIF notebook showed what IIIF does *within one platform*. But there are thousands of libraries, archives and museums, each with its own platform. How does a researcher find objects across all of them?

The usual answer is an **aggregator**: a service that collects descriptions of objects from many institutions into one searchable index. [Europeana](https://www.europeana.eu/) is the largest for European cultural heritage.

In [ ]:
# Setup: run this cell first.
from europeana_helpers import *
from iiif_helpers import url_panel, show_mirador

print(API_KEY_STATUS)
overview, institutions, aggregators, countries = load_overview()   # cached counts; fetched once if missing
total_items = overview["total_items"]

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 2 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## How data gets to Europeana

![Europeana flow: institutions reach Europeana directly, through aggregators, or through projects](../misc/assets/img/europeana_flow.svg)

- Institutions run their own **platforms** and offer **metadata** (for harvesting) and, sometimes, **IIIF** (for images).
- Europeana **harvests the metadata** into its own format and reaches institutions *directly*, *via an aggregator* or *via a project*.
- Europeana is mainly an **index of descriptions**. The full-resolution images stay with the institutions.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 3 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## The Europeana API

An **API** lets a program send a request and receive structured data (JSON) instead of a web page. We use the **Search API**, which can return items, or only *counts* per value of a field.

| API | Purpose |
|---|---|
| **Search API** | Query and filter the whole index; counts per value ("facets") |
| **Record API** | Full metadata of one item |

The whole index, counted by media type (`rows=0`: no items, only counts):

In [ ]:
print(f"Items in the index: {total_items:,}\n")
for t, n in facet_counts("TYPE"):
    print(f"  {t:<6} {n:>12,}   {n / total_items:6.1%}")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 4 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Who contributes?

Counting items per institution (`DATA_PROVIDER`). The bar shows each institution's mix of media types.

In [ ]:
show_table(institutions, sort_by="total", n=12, art=True, iiif=True)

Contributions are very uneven. The counts describe what has been *contributed to Europeana*, not the size of an institution's or country's heritage.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 5 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Routes and countries

The `PROVIDER` facet shows the route by which items arrived: national aggregators such as the German Digital Library (DDB), or projects.

In [ ]:
show_table(aggregators, sort_by="total", n=8, first_col="Provider / aggregator")
show_table(countries, sort_by="total", n=8, first_col="Country")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 6 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## A closer look at one institution

The table says how much an institution holds, not what. A few more facet requests restricted to one institution (`qf=DATA_PROVIDER:"name"`) give a quick profile.

In [ ]:
institution_preview("Wellcome Collection")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 7 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Beyond the website's search box

The website filters well. The API is more useful for *numbers across the whole index* and *stacked filters*, including ones the website does not offer, such as a range on the year, or "records that name a IIIF manifest". Each step narrows the previous one:

In [ ]:
steps = [
    ("query: ship",                                   dict(query="ship")),
    ("+ images only",                                 dict(query="ship", qf=["TYPE:IMAGE"])),
    ("+ openly reusable",                             dict(query="ship", qf=["TYPE:IMAGE"], reusability="open")),
    ("+ dated 1800 to 1899",                          dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]"], reusability="open")),
    ("+ record names a IIIF manifest (lower bound)", dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]", "wr_dcterms_isReferencedBy:*"], reusability="open")),
]
for label, kw in steps:
    print(f"{label:<44} {count(**kw):>10,}")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 8 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## What one record contains

| Field | Meaning |
|---|---|
| `dataProvider` | the institution that holds the object |
| `provider` | the aggregator or route |
| `edmIsShownAt` | the object's page on the institution's website |
| `edmIsShownBy` | the media file on the institution's server |
| `edmPreview` | a small preview image served by Europeana |
| `rights` | the rights statement |

In [ ]:
sample_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE"], theme="art", reusability="open")
sample = sample_results["items"]
item = sample[0]
for field in ["title", "dataProvider", "provider", "country", "type", "rights", "edmIsShownAt", "edmIsShownBy", "edmPreview"]:
    value = item.get(field)
    if isinstance(value, list):
        value = value[0]
    print(f"{field:<14} {value}")